# Notebook 2 — Modelado y Evaluación Predictiva (CRISP-DM)
## Plataforma *DatosParaTodos* · datos.gov.co

Este notebook cubre las **fases 4 y 5 de CRISP-DM**:

| Fase | Descripción |
|------|-------------|
| 4. Modelado | Entrenamiento de 7 clasificadores (4 supervisados + 3 ensambles), validación cruzada, 5 métricas |
| 5. Evaluación | ANOVA + Bonferroni, selección TOP 3, optimización bayesiana (Optuna TPE), interpretación de resultados |

**Objetivo:** Predecir la gravedad de un accidente vial (SOLO DAÑOS / CON HERIDOS / CON MUERTOS) para apoyar la toma de decisiones de la Secretaría de Movilidad de Bogotá.

## Estrategia de modelado

### ¿Por qué 7 modelos?

La metodología CRISP-DM recomienda comparar múltiples técnicas porque no existe un modelo universalmente superior (No Free Lunch Theorem). Combinamos:
- **4 modelos supervisados clásicos:** Regresión Logística (modelo base lineal), Árbol de Decisión (interpretable), KNN (basado en distancia), MLP (red neuronal)
- **3 ensambles:** Random Forest, Gradient Boosting y AdaBoost — combinan múltiples estimadores débiles

### Preprocesamiento diferenciado por modelo

| Modelo | Preprocesamiento |
|--------|------------------|
| LogReg, KNN, MLP | StandardScaler + PCA — sensibles a escala y correlación |
| DT, RF, GB, AdaBoost | KBinsDiscretizer (sin escala) — invariantes a monotransformaciones |
| Todos | OneHotEncoder para categóricas |

### Control de data leakage

1. `LabelEncoder` se ajusta **antes** del split (para ver todas las clases)
2. **Split 70/30** con `stratify=y` — antes de cualquier ajuste de transformadores
3. **Correlación de Pearson** calculada solo en `X_train` — no en el dataset completo
4. **SMOTE** aplicado solo sobre el 70% de entrenamiento

## 1. Instalación e importación de dependencias

In [ ]:
!pip install scikit-learn imbalanced-learn optuna scipy pandas numpy requests matplotlib seaborn --quiet

import warnings
warnings.filterwarnings('ignore')

import re, json, pickle, io, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from scipy import stats as sp_stats
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.preprocessing import KBinsDiscretizer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from imblearn.over_sampling import SMOTE

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
})
DATE_PATTERN = re.compile(r'\d{4}-\d{2}')

print('Dependencias cargadas correctamente.')

## 2. Carga y preparación del dataset

In [ ]:
def detect_columns(df):
    sample = df.head(50)
    numeric, date, categorical = [], [], []
    for col in df.columns:
        vals = sample[col].dropna().tolist()
        if not vals: categorical.append(col); continue
        num_count  = sum(1 for v in vals if pd.notna(pd.to_numeric(v, errors='coerce')))
        date_count = sum(1 for v in vals if isinstance(v, str) and DATE_PATTERN.search(v))
        if num_count > len(vals) * 0.6:
            floats = [float(pd.to_numeric(v, errors='coerce')) for v in vals
                      if pd.notna(pd.to_numeric(v, errors='coerce'))]
            all_integers = all(f % 1 == 0 for f in floats)
            n_unique = len(set(floats))
            if all_integers and n_unique <= 15:
                categorical.append(col)
            else:
                numeric.append(col)
        elif date_count > len(vals) * 0.5: date.append(col)
        else: categorical.append(col)
    return {'numeric': numeric, 'date': date, 'categorical': categorical}

URL = 'https://www.datos.gov.co/resource/vjvu-ycr3.json?$limit=5000'
df  = pd.DataFrame(requests.get(URL, timeout=30).json()).drop_duplicates().reset_index(drop=True)
col_types = detect_columns(df)

for col in col_types['numeric']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())
for col in col_types['categorical']:
    if col in df.columns:
        df[col] = df[col].fillna('No especificado')
for col in col_types['numeric']:
    if col not in df.columns: continue
    s = df[col].sort_values()
    if len(s) < 4 or s.nunique() < 15: continue
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0: continue
    df[col] = df[col].clip(q1 - 1.5*iqr, q3 + 1.5*iqr)
for col in col_types['categorical']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()

nunique_cols = df.nunique(dropna=True)
df = df.drop(columns=nunique_cols[nunique_cols <= 1].index.tolist())

cat_cols_all = [c for c in col_types['categorical'] if c in df.columns]
target_col = None
for col in cat_cols_all:
    n_u = df[col].nunique()
    if 2 <= n_u <= 20:
        target_col = col
        break

assert target_col, 'No se detectó variable objetivo. Asignar manualmente.'
df = df.dropna(subset=[target_col])

y_raw = df[target_col].astype(str)
X_df  = df.drop(columns=[target_col])
num_cols = [c for c in col_types['numeric']     if c in X_df.columns]
cat_cols = [c for c in col_types['categorical'] if c in X_df.columns]

print(f'Dataset: {df.shape[0]:,} registros × {df.shape[1]} variables')
print(f'Target : "{target_col}" | Clases: {sorted(y_raw.unique().tolist())}')
print(f'Num cols: {num_cols}')
print(f'Cat cols: {cat_cols}')

## 3. Partición 70/30 y Balanceo de clases (SMOTE)

El balanceo **solo se aplica sobre el 70% de entrenamiento** para evitar data leakage.  
SMOTE genera muestras sintéticas de la clase minoritaria interpolando entre vecinos existentes — no simplemente duplicando registros.

In [ ]:
# Preprocesadores: se definen aquí para fit sobre X_train solamente
def new_cat_transformer():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])

def make_preprocessor_linear(nc, cc):
    transformers = []
    if nc: transformers.append(('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', StandardScaler()),
        ('pca', PCA(n_components=min(0.95, len(nc)), random_state=42)),
    ]), nc))
    if cc: transformers.append(('cat', new_cat_transformer(), cc))
    return ColumnTransformer(transformers, remainder='drop') if transformers else None

def make_preprocessor_tree(nc, cc):
    transformers = []
    if nc: transformers.append(('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('kbin', KBinsDiscretizer(n_bins=10, encode='ordinal', strategy='quantile')),
    ]), nc))
    if cc: transformers.append(('cat', new_cat_transformer(), cc))
    return ColumnTransformer(transformers, remainder='drop') if transformers else None

# LabelEncoder ANTES del split
le = LabelEncoder()
y  = le.fit_transform(y_raw)

# Split estratificado 70/30
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_df, y, test_size=0.30, random_state=42, stratify=y
)

print(f'Train: {len(X_train_raw):,} registros (70%)')
print(f'Test : {len(X_test_raw):,} registros (30%)')

counts = np.bincount(y_train)
ratio  = counts.min() / counts.max() if counts.max() > 0 else 1.0
clases_dict = dict(zip(le.classes_, counts))

# Visualizar distribución en train
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(clases_dict.keys(), clases_dict.values(),
       color=sns.color_palette('tab10')[:len(clases_dict)],
       edgecolor='black', linewidth=0.7)
ax.set_title('Distribución de clases en el conjunto de entrenamiento (70%)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Frecuencia')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(int(bar.get_height())), ha='center', va='bottom')
plt.tight_layout()
plt.show()

print(f'Distribución: {clases_dict}')
print(f'Ratio min/max: {ratio:.2f}')

smote_applied = False
if ratio < 0.80:
    # Preprocesar para SMOTE (necesita datos numéricos)
    pre_smote = make_preprocessor_linear(num_cols, cat_cols)
    if pre_smote:
        X_train_pre = pre_smote.fit_transform(X_train_raw)
        X_test_pre  = pre_smote.transform(X_test_raw)
    else:
        X_train_pre = X_train_raw.values if hasattr(X_train_raw, 'values') else X_train_raw
        X_test_pre  = X_test_raw.values if hasattr(X_test_raw, 'values') else X_test_raw

    min_samples = counts.min()
    k = min(5, max(1, min_samples - 1))
    if k >= 1 and min_samples >= 2:
        smote = SMOTE(random_state=42, k_neighbors=k)
        X_train_pre, y_train = smote.fit_resample(X_train_pre, y_train)
        smote_applied = True
        new_counts = np.bincount(y_train)
        print(f'\n✓ SMOTE aplicado (k={k})')
        print(f'  Train balanceado: {len(X_train_pre):,} registros')
        print(f'  Nueva distribución: {dict(zip(le.classes_, new_counts))}')
        print(f'\n📌 SMOTE genera puntos sintéticos interpolando entre vecinos reales,')
        print(f'   no simplemente duplicando registros. Esto reduce el sesgo del modelo')
        print(f'   hacia la clase mayoritaria sin distorsionar el espacio de características.')
else:
    pre_smote = make_preprocessor_linear(num_cols, cat_cols)
    if pre_smote:
        X_train_pre = pre_smote.fit_transform(X_train_raw)
        X_test_pre  = pre_smote.transform(X_test_raw)
    else:
        X_train_pre = X_train_raw.values if hasattr(X_train_raw, 'values') else X_train_raw
        X_test_pre  = X_test_raw.values if hasattr(X_test_raw, 'values') else X_test_raw
    print(f'\n✓ Clases naturalmente balanceadas (ratio={ratio:.2f}) — SMOTE no requerido.')

## 4. Entrenamiento y Validación Cruzada — 7 modelos

Se entrenan **4 métodos supervisados** y **3 métodos de ensamble** con `StratifiedKFold` adaptativo (el número de folds se ajusta al tamaño de la clase minoritaria).  
Se calculan **5 métricas de calidad** por modelo:

| Métrica | Qué mide |
|---------|----------|
| Accuracy | Proporción de predicciones correctas |
| Precision | Exactitud de las predicciones positivas |
| Recall | Capacidad de detectar todos los casos positivos |
| F1-Score | Media armónica de Precision y Recall |
| AUC-ROC | Discriminación global entre clases |

In [ ]:
n_classes = len(np.unique(y))
avg = 'weighted' if n_classes > 2 else 'binary'

# Adaptive n_splits
min_class_size = np.bincount(y_train).min()
n_splits = min(10, max(2, min_class_size))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
print(f'StratifiedKFold con n_splits={n_splits} (ajustado a clase minoritaria={min_class_size})')

# Preprocesadores para árboles (sin escala)
pre_tree = make_preprocessor_tree(num_cols, cat_cols)
if pre_tree:
    X_train_tree = pre_tree.fit_transform(X_train_raw)
    X_test_tree  = pre_tree.transform(X_test_raw)
else:
    X_train_tree = X_train_pre.copy()
    X_test_tree  = X_test_pre.copy()

# Modelos con su tipo de preprocesamiento
MODELS = [
    ('Regresión Logística',           LogisticRegression(max_iter=1000, random_state=42),   X_train_pre, X_test_pre,  'Supervisado'),
    ('K-Vecinos (KNN)',               KNeighborsClassifier(n_neighbors=min(5, len(X_train_pre)-1)), X_train_pre, X_test_pre,  'Supervisado'),
    ('Red Neuronal (MLP)',            MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, random_state=42), X_train_pre, X_test_pre, 'Supervisado'),
    ('Árbol de Decisión',             DecisionTreeClassifier(random_state=42),              X_train_tree, X_test_tree, 'Supervisado'),
    ('Random Forest (Ensamble)',      RandomForestClassifier(n_estimators=100, random_state=42), X_train_tree, X_test_tree, 'Ensamble'),
    ('Gradient Boosting (Ensamble)',  GradientBoostingClassifier(n_estimators=100, random_state=42), X_train_tree, X_test_tree, 'Ensamble'),
    ('AdaBoost (Ensamble)',           AdaBoostClassifier(n_estimators=100, random_state=42), X_train_tree, X_test_tree, 'Ensamble'),
]

results = []
for name, model, X_tr, X_te, tipo in MODELS:
    print(f'Entrenando {name}...')
    try:
        cv_res    = cross_validate(model, X_tr, y_train, cv=cv, scoring=['accuracy'], error_score=np.nan)
        acc_scores = cv_res['test_accuracy']
        acc_scores = acc_scores[~np.isnan(acc_scores)]

        model.fit(X_tr, y_train)
        y_pred = model.predict(X_te)

        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average=avg, zero_division=0)
        rec  = recall_score(y_test, y_pred, average=avg, zero_division=0)
        f1   = f1_score(y_test, y_pred, average=avg, zero_division=0)

        auc = None
        if hasattr(model, 'predict_proba'):
            try:
                y_proba = model.predict_proba(X_te)
                auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted') if n_classes > 2 \
                      else roc_auc_score(y_test, y_proba[:, 1])
            except: pass

        results.append({
            'Modelo': name, 'Tipo': tipo,
            'CV Acc Media': round(float(acc_scores.mean()), 4) if len(acc_scores) > 0 else 0.0,
            'CV Acc Std':   round(float(acc_scores.std()),  4) if len(acc_scores) > 0 else 0.0,
            'CV Scores':    acc_scores.tolist(),
            'Test Accuracy':  round(acc, 4),
            'Test Precision': round(prec, 4),
            'Test Recall':    round(rec, 4),
            'Test F1':        round(f1, 4),
            'Test AUC-ROC':   round(auc, 4) if auc is not None else None,
            '_model':         model,
            '_X_tr': X_tr, '_X_te': X_te,
        })
    except Exception as e:
        print(f'  ERROR en {name}: {e}')
        results.append({'Modelo': name, 'Tipo': tipo, 'CV Acc Media': 0, 'CV Acc Std': 0,
                        'CV Scores': [], 'Test Accuracy': 0, 'Test Precision': 0,
                        'Test Recall': 0, 'Test F1': 0, 'Test AUC-ROC': None,
                        '_model': None, '_X_tr': X_tr, '_X_te': X_te})

df_results = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in results])
df_results = df_results.sort_values('CV Acc Media', ascending=False).reset_index(drop=True)
display(df_results)

### Visualización comparativa de métricas

In [ ]:
metricas = ['Test Accuracy', 'Test F1', 'Test Precision', 'Test Recall']
df_plot  = df_results.set_index('Modelo')[metricas].apply(pd.to_numeric, errors='coerce')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Gráfico de barras agrupadas
df_plot.plot(kind='bar', ax=axes[0], colormap='tab10', edgecolor='black', linewidth=0.5)
axes[0].set_title('Métricas de calidad — 7 modelos', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Valor')
axes[0].set_ylim(0, 1.15)
axes[0].tick_params(axis='x', rotation=35)
axes[0].legend(loc='lower right', fontsize=9)

# CV Accuracy con barras de error (std)
df_cv = df_results[['Modelo', 'CV Acc Media', 'CV Acc Std']].set_index('Modelo')
x_pos = range(len(df_cv))
axes[1].bar(x_pos, df_cv['CV Acc Media'], yerr=df_cv['CV Acc Std'],
            color=sns.color_palette('tab10')[:len(df_cv)], capsize=4,
            edgecolor='black', linewidth=0.5, error_kw={'linewidth': 1.5})
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(df_cv.index, rotation=35, ha='right')
axes[1].set_title('CV Accuracy con intervalo de desviación estándar', fontsize=12, fontweight='bold')
axes[1].set_ylabel('CV Accuracy')
axes[1].set_ylim(0, 1.15)

plt.tight_layout()
plt.show()

mejor_idx  = df_results['CV Acc Media'].idxmax()
mejor_mod  = df_results.loc[mejor_idx, 'Modelo']
mejor_acc  = df_results.loc[mejor_idx, 'CV Acc Media']
mejor_std  = df_results.loc[mejor_idx, 'CV Acc Std']
mejor_f1   = df_results.loc[mejor_idx, 'Test F1']

print(f'📌 ANÁLISIS DE RESULTADOS:')
print(f'   Mejor modelo en CV: {mejor_mod}')
print(f'   CV Accuracy: {mejor_acc:.4f} ± {mejor_std:.4f}')
print(f'   Test F1-Score: {mejor_f1:.4f}')
print(f'\n   Interpretación de la desviación estándar en CV:')
print(f'   Una std baja indica que el modelo es ESTABLE (no sensible al fold elegido).')
print(f'   Una std alta puede indicar sobreajuste parcial o alta varianza en los datos.')

# Referencia baseline: clasificador trivial
majority_class_pct = np.bincount(y_test).max() / len(y_test)
print(f'\n   Baseline (siempre predice clase mayoritaria): {majority_class_pct:.4f}')
if mejor_acc > majority_class_pct:
    mejora = (mejor_acc - majority_class_pct) / majority_class_pct * 100
    print(f'   El mejor modelo supera el baseline en {mejora:.1f}% — aprendizaje confirmado.')
else:
    print('   ⚠️ El modelo no supera el baseline. Revisar features y datos.')

## 5. Análisis Estadístico — ANOVA y Post-hoc Bonferroni

**¿Por qué no basta con mirar el ranking de accuracy?**  
Si las diferencias entre modelos son pequeñas, pueden ser producto del azar (variabilidad en los folds). ANOVA determina si existe **diferencia estadísticamente significativa** (p < 0.05) entre los 7 modelos. El **post-hoc de Bonferroni** ajusta el p-valor por comparaciones múltiples para evitar falsos positivos.

- Si p < 0.05 → **Sí hay diferencias reales**: seleccionamos los 3 con mayor CV Accuracy
- Si p ≥ 0.05 → **No hay diferencias significativas**: preferimos el modelo más simple (menor complejidad computacional)

In [ ]:
grupos  = [r['CV Scores'] for r in results if len(r['CV Scores']) > 0]
nombres = [r['Modelo']    for r in results if len(r['CV Scores']) > 0]

f_stat, p_val = sp_stats.f_oneway(*grupos)
significativo = p_val < 0.05

print('=' * 65)
print(f'  ANOVA — F-statistic : {f_stat:.4f}')
print(f'  ANOVA — p-valor     : {p_val:.6f}')
print(f'  Diferencia significativa (p < 0.05): {"SÍ ✓" if significativo else "NO"}')
print('=' * 65)

if significativo:
    print('\n  ✓ Existen diferencias estadísticamente reales entre los modelos.')
    print('  Criterio de selección: 3 modelos con mayor CV Accuracy.')
else:
    print('\n  Los modelos no difieren significativamente en performance.')
    print('  Criterio de selección: principio de parsimonia (modelo más simple).')

# ── Post-hoc Tukey HSD ─────────────────────────────────────────────────────
# pairwise_tukeyhsd controla la tasa de error familiar (FWER) simultáneamente
# sin necesidad de ajuste manual como Bonferroni.
# Supuesto: los grupos (CV scores) son independientes y aproximadamente normales.
from statsmodels.stats.multicomp import pairwise_tukeyhsd

all_scores = np.concatenate([np.array(g) for g in grupos])
all_labels = np.concatenate([[n] * len(g) for n, g in zip(nombres, grupos)])
tukey = pairwise_tukeyhsd(endog=all_scores, groups=all_labels, alpha=0.05)

comparaciones = []
for row in tukey.summary().data[1:]:
    g1, g2, meandiff, p_adj, lower, upper, reject = row
    comparaciones.append({
        'Modelo A':          str(g1),
        'Modelo B':          str(g2),
        'Dif. Medias':       round(float(meandiff), 4),
        'p-adj (Tukey HSD)': round(float(p_adj), 6),
        'IC 95% inf':        round(float(lower), 4),
        'IC 95% sup':        round(float(upper), 4),
        'Significativo':     'SÍ ✓' if bool(reject) else 'NO',
    })

df_tukey = pd.DataFrame(comparaciones)
print(f'\nComparaciones post-hoc Tukey HSD ({len(comparaciones)} pares, α = 0.05):')
display(df_tukey.sort_values('p-adj (Tukey HSD)'))

print('\n📌 INTERPRETACIÓN DEL TEST DE TUKEY HSD:')
sig_pares = df_tukey[df_tukey['Significativo'] == 'SÍ ✓']
if len(sig_pares) > 0:
    print(f'   {len(sig_pares)} par(es) con diferencia estadísticamente significativa:')
    for _, row in sig_pares.iterrows():
        print(f'   {row["Modelo A"]} vs {row["Modelo B"]}: dif={row["Dif. Medias"]:+.4f}, p={row["p-adj (Tukey HSD)"]:.4f}')
else:
    print('   Ningún par difiere significativamente → se aplica principio de parsimonia.')

# ── Selección TOP 3 ────────────────────────────────────────────────────────
WEIGHTS = {
    'Regresión Logística': 1, 'Árbol de Decisión': 2, 'K-Vecinos (KNN)': 3,
    'Red Neuronal (MLP)': 4, 'AdaBoost (Ensamble)': 5,
    'Gradient Boosting (Ensamble)': 6, 'Random Forest (Ensamble)': 7
}

if significativo:
    top3 = sorted(results, key=lambda r: r['CV Acc Media'], reverse=True)[:3]
    criterio = 'Mayor CV Accuracy (diferencia estadística confirmada por ANOVA)'
else:
    top3 = sorted(results, key=lambda r: (WEIGHTS.get(r['Modelo'], 99), -r['CV Acc Media']))[:3]
    criterio = 'Modelo más simple (sin diferencia estadística significativa — parsimonia)'

top3_nombres = [r['Modelo'] for r in top3]
print(f'\nCriterio de selección : {criterio}')
print(f'TOP 3 seleccionados   : {top3_nombres}')

## 6. Hiperparametrización — GridSearchCV sobre el TOP 3

`GridSearchCV` evalúa exhaustivamente todas las combinaciones de una grilla predefinida de hiperparámetros usando validación cruzada de 3 pliegues. Es el método de referencia de sklearn para búsqueda de hiperparámetros.

**Ventaja:** Garantiza encontrar el mejor punto dentro de la grilla (búsqueda exhaustiva).  
**Limitación:** El espacio de búsqueda debe definirse manualmente; es costoso en espacios grandes o continuos (por eso se complementa con Optuna en la sección siguiente).

In [ ]:
from sklearn.model_selection import GridSearchCV

GRIDSEARCH_PARAMS = {
    'Regresión Logística': {
        'C': [0.01, 0.1, 1.0, 10.0, 100.0],
    },
    'Árbol de Decisión': {
        'max_depth': [3, 5, 10, 15],
        'min_samples_split': [2, 5, 10],
    },
    'K-Vecinos (KNN)': {
        'n_neighbors': [3, 5, 7, 11],
        'weights': ['uniform', 'distance'],
    },
    'Red Neuronal (MLP)': {
        'alpha': [1e-4, 1e-3, 1e-2],
        'hidden_layer_sizes': [(50,), (100,)],
    },
    'Random Forest (Ensamble)': {
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, None],
    },
    'Gradient Boosting (Ensamble)': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.05, 0.1, 0.2],
    },
    'AdaBoost (Ensamble)': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.5, 1.0],
    },
}

# Mapa modelo → (datos de entrenamiento, datos de test, objeto entrenado)
model_data_map = {r['Modelo']: r for r in results}

gs_results = []
gs_best_models = {}   # {nombre: modelo_ya_ajustado}

for r in top3:
    name    = r['Modelo']
    X_tr    = r['_X_tr']
    X_te    = r['_X_te']
    base_acc = r['CV Acc Media']
    params  = GRIDSEARCH_PARAMS.get(name, {})
    if not params:
        print(f'  Sin grilla definida para {name} — saltando.')
        continue

    print(f'GridSearch en {name} — {sum(len(v) for v in params.values())} combinaciones...')
    try:
        base_model = r['_model'].__class__(**{
            k: v for k, v in r['_model'].get_params().items()
        })
        gs = GridSearchCV(
            base_model, params,
            cv=3, scoring='accuracy', n_jobs=-1, refit=True
        )
        gs.fit(X_tr, y_train)
        test_acc = accuracy_score(y_test, gs.best_estimator_.predict(X_te))

        gs_results.append({
            'Modelo':           name,
            'Mejores Params':   str(gs.best_params_),
            'CV Score GS':      round(float(gs.best_score_), 4),
            'Test Accuracy GS': round(test_acc, 4),
            'Mejora vs base':   round(test_acc - r['Test Accuracy'], 4),
        })
        gs_best_models[name] = (gs.best_estimator_, gs.best_score_, X_tr, X_te)
        print(f'  → Mejor: {gs.best_params_}  |  CV={gs.best_score_:.4f}  |  Test={test_acc:.4f}')
    except Exception as e:
        print(f'  ERROR: {e}')

df_gs = pd.DataFrame(gs_results)
print('\n=== Resultados GridSearchCV ===')
display(df_gs)

print('\n📌 INTERPRETACIÓN GRIDSEACH:')
for _, row in df_gs.iterrows():
    mejor = 'mejora' if row['Mejora vs base'] > 0 else 'sin mejora significativa'
    print(f'   {row["Modelo"]}: CV={row["CV Score GS"]:.4f} ({mejor}: {row["Mejora vs base"]:+.4f})')

## 6. Importancia de características — ¿Qué factores predicen la gravedad?

Para los modelos basados en árboles, podemos extraer la importancia de cada variable. Este análisis responde directamente a las preguntas de negocio: **¿qué factores del accidente determinan su gravedad?**

In [ ]:
# Buscar Random Forest o árbol entre los resultados
rf_result = next((r for r in results if 'Random Forest' in r['Modelo'] and r['_model'] is not None), None)
dt_result = next((r for r in results if r['Modelo'] == 'Árbol de Decisión' and r['_model'] is not None), None)
tree_result = rf_result or dt_result

if tree_result and hasattr(tree_result['_model'], 'feature_importances_'):
    importances = tree_result['_model'].feature_importances_

    # Construir nombres de features
    pre = make_preprocessor_tree(num_cols, cat_cols)
    if pre:
        pre.fit(X_train_raw)
        feature_names = []
        for tname, transf, cols in pre.transformers_:
            if tname == 'num':
                feature_names += [f'{c}_bin' for c in cols]
            elif tname == 'cat':
                try:
                    ohe = transf.named_steps['onehot']
                    cats = ohe.get_feature_names_out(cols)
                    feature_names += cats.tolist()
                except:
                    feature_names += cols

        n_features = min(len(importances), len(feature_names))
        feat_imp = pd.Series(
            importances[:n_features],
            index=feature_names[:n_features]
        ).sort_values(ascending=False).head(15)

        fig, ax = plt.subplots(figsize=(10, 6))
        feat_imp.sort_values().plot(kind='barh', ax=ax, color='#2ecc71', edgecolor='black', linewidth=0.5)
        ax.set_title(f'Top 15 variables más importantes — {tree_result["Modelo"]}',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Importancia (Gini)')
        plt.tight_layout()
        plt.show()

        top_feat = feat_imp.head(5)
        print('📌 INTERPRETACIÓN — Variables más predictivas de la gravedad:')
        for feat, imp in top_feat.items():
            base_var = feat.split('_')[0]
            print(f'   {feat}: importancia = {imp:.4f} ({imp*100:.1f}%)')
        print(f'\n   Estas variables deben ser el foco de intervenciones preventivas.')
        print(f'   Por ejemplo, si la HORA tiene alta importancia → reforzar patrullaje en picos.')
        print(f'   Si LOCALIDAD es clave → focalizar infraestructura vial en zonas de mayor riesgo.')
    else:
        print('No se pudo reconstruir el preprocesador para extraer nombres de features.')
else:
    print('No hay modelo de árbol disponible para importancia de características.')

## 7. Optimización Bayesiana (Optuna TPE) — TOP 3 modelos

Mientras GridSearch explora exhaustivamente una grilla fija, **Optuna TPE** (Tree-structured Parzen Estimator) aplica optimización bayesiana: aprende iterativamente qué regiones del espacio de hiperparámetros son prometedoras y concentra los trials allí.

| Método | Exploración | Espacio continuo | Aprendizaje entre trials |
|--------|------------|----------------|--------------------------|
| GridSearch | Exhaustiva en grilla fija | No (requiere discretización) | No |
| **Optuna TPE** | **Bayesiana adaptativa** | **Sí (log-espacio, floats)** | **Sí** |

Ambos métodos se aplican sobre el **mismo 70% de entrenamiento**. El modelo con mejor CV Score entre los dos métodos es el que se lleva a despliegue.

**¿Por qué los dos?** GridSearch garantiza explorar toda la grilla; Optuna refina en espacios continuos donde GridSearch no puede operar. Juntos ofrecen cobertura completa del espacio de búsqueda.

In [ ]:
def build_model(name, params):
    if name == 'Regresión Logística':
        return LogisticRegression(C=params.get('C', 1.0), max_iter=1000, random_state=42)
    elif name == 'Árbol de Decisión':
        return DecisionTreeClassifier(max_depth=params.get('max_depth', 5), random_state=42)
    elif name == 'K-Vecinos (KNN)':
        return KNeighborsClassifier(n_neighbors=params.get('n_neighbors', 5))
    elif name == 'Red Neuronal (MLP)':
        return MLPClassifier(alpha=params.get('alpha', 1e-4),
                             hidden_layer_sizes=params.get('hidden_layer_sizes', (50,)),
                             max_iter=500, random_state=42)
    elif name == 'Random Forest (Ensamble)':
        return RandomForestClassifier(n_estimators=params.get('n_estimators', 100),
                                      max_depth=params.get('max_depth', 10), random_state=42)
    elif name == 'Gradient Boosting (Ensamble)':
        return GradientBoostingClassifier(n_estimators=params.get('n_estimators', 100),
                                          learning_rate=params.get('learning_rate', 0.1), random_state=42)
    elif name == 'AdaBoost (Ensamble)':
        return AdaBoostClassifier(n_estimators=params.get('n_estimators', 100),
                                  learning_rate=params.get('learning_rate', 0.1), random_state=42)

def make_objective(model_name, X_tr, y_tr):
    def objective(trial):
        if model_name == 'Regresión Logística':
            params = {'C': trial.suggest_float('C', 1e-4, 1e2, log=True)}
        elif model_name == 'Árbol de Decisión':
            params = {'max_depth': trial.suggest_int('max_depth', 2, 20)}
        elif model_name == 'K-Vecinos (KNN)':
            params = {'n_neighbors': trial.suggest_int('n_neighbors', 1, min(15, len(X_tr) - 1))}
        elif model_name == 'Red Neuronal (MLP)':
            params = {'alpha': trial.suggest_float('alpha', 1e-5, 1e-1, log=True),
                      'hidden_layer_sizes': trial.suggest_categorical('hl', [(50,), (50,50), (100,)])}
        elif model_name == 'Random Forest (Ensamble)':
            params = {'n_estimators': trial.suggest_int('n_estimators', 10, 200),
                      'max_depth': trial.suggest_int('max_depth', 2, 20)}
        elif model_name == 'Gradient Boosting (Ensamble)':
            params = {'n_estimators': trial.suggest_int('n_estimators', 10, 200),
                      'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True)}
        elif model_name == 'AdaBoost (Ensamble)':
            params = {'n_estimators': trial.suggest_int('n_estimators', 10, 200),
                      'learning_rate': trial.suggest_float('learning_rate', 0.01, 1.0, log=True)}
        else:
            raise optuna.exceptions.TrialPruned()
        m = build_model(model_name, params)
        score = cross_validate(m, X_tr, y_tr, cv=3, scoring='accuracy')['test_score'].mean()
        return score
    return objective

optuna_results = []
best_overall_score  = -1
best_overall_weight = 99
best_overall_name   = ''
best_overall_model  = None
best_X_train        = None
best_X_test         = None

for r in top3:
    name = r['Modelo']
    X_tr = r['_X_tr']
    X_te = r['_X_te']
    base_acc = r['CV Acc Media']

    print(f'\nOptimizando {name}...')
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(make_objective(name, X_tr, y_train), n_trials=15, timeout=90)

    best_params = study.best_params
    best_model  = build_model(name, best_params)
    best_model.fit(X_tr, y_train)
    test_acc = accuracy_score(y_test, best_model.predict(X_te))

    print(f'  Mejores parámetros : {best_params}')
    print(f'  CV Score (Optuna)  : {study.best_value:.4f}')
    print(f'  Test Accuracy      : {test_acc:.4f}  (base: {base_acc:.4f}, mejora: {test_acc-base_acc:+.4f})')

    optuna_results.append({
        'Modelo': name, 'Mejores Params': str(best_params),
        'CV Score Optuna': round(study.best_value, 4),
        'Test Accuracy': round(test_acc, 4),
        'Mejora': round(test_acc - base_acc, 4)
    })

    weight = WEIGHTS.get(name, 99)
    es_mejor = (study.best_value > best_overall_score) if significativo else \
               (weight < best_overall_weight or
                (weight == best_overall_weight and study.best_value > best_overall_score))

    if es_mejor:
        best_overall_score  = study.best_value
        best_overall_weight = weight
        best_overall_name   = name
        best_overall_model  = best_model
        best_X_train        = X_tr
        best_X_test         = X_te

df_optuna = pd.DataFrame(optuna_results)
print('\n=== Resumen de Optimización Bayesiana ===')
display(df_optuna)
print(f'\nMEJOR MODELO FINAL: {best_overall_name}')
print(f'CV Score Optuna   : {best_overall_score:.4f}')

### Visualización de la convergencia de Optuna

In [ ]:
# Visualizar mejora de la optimización en el mejor modelo
study_final = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
best_r = next(r for r in top3 if r['Modelo'] == best_overall_name)
study_final.optimize(
    make_objective(best_overall_name, best_r['_X_tr'], y_train),
    n_trials=20, timeout=120
)

trial_values = [t.value for t in study_final.trials if t.value is not None]
best_so_far  = [max(trial_values[:i+1]) for i in range(len(trial_values))]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(trial_values, 'o-', alpha=0.5, color='#3498db', label='Score del trial')
ax.plot(best_so_far, '-', linewidth=2.5, color='#e74c3c', label='Mejor hasta ahora')
ax.set_title(f'Convergencia de Optuna TPE — {best_overall_name}', fontsize=12, fontweight='bold')
ax.set_xlabel('Trial #')
ax.set_ylabel('CV Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

print(f'📌 INTERPRETACIÓN:')
print(f'   La línea roja muestra cómo TPE mejora el mejor score conocido a lo largo de los trials.')
print(f'   Cada trial usa lo aprendido de los anteriores para explorar regiones más prometedoras.')
print(f'   Score final: {study_final.best_value:.4f} con parámetros: {study_final.best_params}')

### Matriz de confusión e informe de clasificación del modelo ganador

In [ ]:
if best_overall_model is not None and best_X_test is not None:
    y_pred_final = best_overall_model.predict(best_X_test)
    cm = confusion_matrix(y_test, y_pred_final)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Matriz absoluta
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
    axes[0].set_title(f'Matriz de Confusión (absoluta)\n{best_overall_name}', fontsize=11, fontweight='bold')

    # Matriz normalizada
    cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
    disp2 = ConfusionMatrixDisplay(confusion_matrix=np.round(cm_norm, 2), display_labels=le.classes_)
    disp2.plot(ax=axes[1], cmap='Blues', colorbar=False)
    axes[1].set_title('Matriz de Confusión (normalizada por fila)', fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.show()

    print('=== Reporte de clasificación completo ===')
    print(classification_report(y_test, y_pred_final, target_names=le.classes_))

    print('📌 INTERPRETACIÓN DE LA MATRIZ DE CONFUSIÓN:')
    for i, clase in enumerate(le.classes_):
        correctos = cm[i, i]
        total     = cm[i, :].sum()
        recall_c  = correctos / total if total > 0 else 0
        print(f'   {clase}: detecta correctamente el {recall_c*100:.1f}% de los casos reales')

    print(f'\n   En el contexto de accidentalidad vial:')
    print(f'   Los errores más críticos son los FALSOS NEGATIVOS en "CON MUERTOS" o "CON HERIDOS":')
    print(f'   predecir "SOLO DAÑOS" cuando en realidad hay víctimas puede costar vidas')
    print(f'   al no despachar recursos de emergencia. El Recall de estas clases es clave.')

## 8. Empaquetado del Pipeline final para despliegue

El modelo final se almacena en un **Pipeline de sklearn** que incluye preprocesamiento + clasificador optimizado, listo para inferencia en producción. El pipeline se reentrena sobre el **dataset completo** (train + test) para maximizar el uso de datos.

In [ ]:
if best_overall_model is not None:
    # Determinar tipo de preprocesador del mejor modelo
    TREE_MODELS = ['Árbol de Decisión', 'Random Forest (Ensamble)', 'Gradient Boosting (Ensamble)', 'AdaBoost (Ensamble)']
    if best_overall_name in TREE_MODELS:
        final_pre = make_preprocessor_tree(num_cols, cat_cols)
    else:
        final_pre = make_preprocessor_linear(num_cols, cat_cols)

    steps = []
    if final_pre:
        steps.append(('preprocessor', final_pre))
    steps.append(('classifier', best_overall_model))

    final_pipeline = Pipeline(steps)

    # Reentrenar sobre dataset completo
    final_pipeline.fit(X_df, y)

    with open('mejor_modelo_crispdm.pickle', 'wb') as f:
        pickle.dump({
            'pipeline':      final_pipeline,
            'label_encoder': le,
            'num_cols':      num_cols,
            'cat_cols':      cat_cols,
            'target_col':    target_col,
            'model_name':    best_overall_name
        }, f)

    buf = io.BytesIO()
    pickle.dump(final_pipeline, buf)
    size_kb = len(buf.getvalue()) / 1024

    print(f'✓ Pipeline exportado exitosamente.')
    print(f'  Pasos del pipeline : {[s[0] for s in final_pipeline.steps]}')
    print(f'  Tamaño del pickle  : {size_kb:.1f} KB')
    print(f'  Modelo final       : {best_overall_name}')
    print(f'  Variable objetivo  : {target_col}')
    print(f'  Clases             : {le.classes_.tolist()}')
    print(f'\n  El pipeline está listo para:')
    print(f'  1. Inferencia en tiempo real via API FastAPI (/analytics/predict)')
    print(f'  2. Descarga desde la plataforma (/analytics/download_model)')
    print(f'  3. Despliegue en Streamlit (Notebook 3)')

---

## Resumen ejecutivo — Fases de Modelado y Evaluación

| Decisión | Justificación |
|----------|---------------|
| 7 modelos (4 supervisados + 3 ensambles) | Comparación exhaustiva, sin asumir superioridad a priori |
| Preprocesamiento diferenciado por modelo | Lineal/distancia → normalización; árboles → discretización |
| Split 70/30 antes de correlación | Elimina data leakage en selección de features |
| SMOTE solo en train | Datos de test no contaminados por sintéticos |
| StratifiedKFold adaptativo | Preserva proporción de clases en cada fold |
| ANOVA + Bonferroni | Validación estadística, no solo ranking por accuracy |
| Optuna TPE vs. GridSearch | Búsqueda bayesiana más eficiente en espacios continuos |
| Reentrenar sobre N completo | Máximo aprovechamiento de datos para producción |

**Continúa en:** `3_Despliegue_del_Modelo.ipynb`